L'objectif est de choisir une année pour le clustering et choisir les pays à conserver

In [1]:
import pandas as pd
from pathlib import Path

In [83]:
GDP_COLUMNS = ['gdp_per_capita_ppp_current_ppp', 'gdp_per_capita_usd_current_usd']


def select_gdp(df, gdp_column):
    """Keep gdp_column and drop the other GDP columns."""
    other_gdp_columns = [col for col in GDP_COLUMNS if col != gdp_column]
    return df.drop(columns=other_gdp_columns)


def split_by_completeness(df, year, gdp_column):
    """Return (df_full, df_missing) for one year: countries with every feature filled vs the others."""
    df_year = select_gdp(df[df['time_period'] == year], gdp_column)
    is_full = df_year.notna().all(axis=1)
    return df_year[is_full], df_year[~is_full]


def missing_population_report(df, year, gdp_column):
    """Missing countries for one year, sorted by their share of the missing population."""
    _, report = split_by_completeness(df, year, gdp_column)
    report = report.copy()
    world_population = df.loc[df['time_period'] == year, 'population_ps'].sum()
    report['percent_of_missing_population'] = 100 * report['population_ps'] / report['population_ps'].sum()
    report['percent_of_world_population'] = 100 * report['population_ps'] / world_population
    report = report.sort_values('percent_of_missing_population', ascending=False).reset_index(drop=True)
    first_columns = ['name', 'population_ps', 'percent_of_missing_population', 'percent_of_world_population']
    feature_columns = [col for col in report.columns if col not in first_columns + ['ref_area', 'time_period']]
    return report[first_columns + feature_columns]

In [2]:
df = pd.read_csv('../data/interim/interim_dataset.csv')
df.head()

,ref_area,time_period,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,gdp_per_capita_usd_current_usd,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
0,ABW,2000,90588,180.0,30245.706974,20681.023027,72.939,2.968384,NaN,65.354550,NaN
1,ABW,2001,91439,180.0,31920.239073,20740.132583,73.044,2.973567,NaN,65.335114,NaN
2,ABW,2002,92074,180.0,31888.508737,21307.248251,73.135,3.225666,NaN,65.282069,NaN
3,ABW,2003,93128,180.0,32507.084315,21949.485996,73.236,3.676660,NaN,65.198292,NaN
4,ABW,2004,95138,180.0,35059.273098,23700.631990,73.223,3.672560,NaN,65.088653,NaN


In [9]:
features_columns = [col for col in df.columns if col not in ['ref_area', 'time_period', 'name']]
print(df[features_columns].isna().any()[df[features_columns].isna().any()])

land_area_km2                                  True
gdp_per_capita_ppp_current_ppp                 True
gdp_per_capita_usd_current_usd                 True
co2_emissions_per_capita_t                     True
pm2_5_exposure_mcg_m3                          True
health_expenditure_percentage_of_gdp_pt_gdp    True
dtype: bool


In [86]:
df_count_per_year = df.groupby('time_period').count()
df_count_per_year

,ref_area,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,gdp_per_capita_usd_current_usd,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
time_period,,,,,,,,,,
2000,217,217,209,194,204,217,203,200,217,184
2001,217,217,209,194,205,217,203,200,217,184
2002,217,217,209,195,209,217,203,200,217,185
2003,217,217,209,195,209,217,203,200,217,187
2004,217,217,209,195,209,217,203,200,217,187
2005,217,217,209,195,209,217,203,200,217,187
2006,217,217,211,196,210,217,203,200,217,187
2007,217,217,211,197,210,217,203,200,217,187
2008,217,217,211,200,211,217,203,200,217,187


- I want a recent year (more than 2020)
- 2020 and 2021 are to exposed to covid 19
- 2024 has many missing values
- 2022 and 2023 are good candidates

Let's check the number of countries with full data and the countries that are not

In [87]:
df_area = pd.read_csv(Path('../data/external/area_data.csv'))
df_countries = df_area[df_area['region.value'] != 'Aggregates']
df_countries_names = df_countries[['id', 'name']].rename(columns={'id': 'ref_area'})
df_named = df.merge(df_countries_names, on='ref_area', how='left')
df_named = df_named[['name'] + list(df_named.columns[:-1])]
df_named

,name,ref_area,time_period,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,gdp_per_capita_usd_current_usd,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
0,Aruba,ABW,2000,90588,180.0,30245.706974,20681.023027,72.939,2.968384,NaN,65.354550,NaN
1,Aruba,ABW,2001,91439,180.0,31920.239073,20740.132583,73.044,2.973567,NaN,65.335114,NaN
2,Aruba,ABW,2002,92074,180.0,31888.508737,21307.248251,73.135,3.225666,NaN,65.282069,NaN
3,Aruba,ABW,2003,93128,180.0,32507.084315,21949.485996,73.236,3.676660,NaN,65.198292,NaN
4,Aruba,ABW,2004,95138,180.0,35059.273098,23700.631990,73.223,3.672560,NaN,65.088653,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
5420,Zimbabwe,ZWE,2020,15526888,386850.0,4178.585579,2059.637040,61.530,0.584444,13.698771,37.009767,2.954401
5421,Zimbabwe,ZWE,2021,15797210,386850.0,4827.088694,2613.616741,60.135,0.688925,15.144709,37.869206,2.671119
5422,Zimbabwe,ZWE,2022,16069056,386850.0,5395.856298,2536.401971,62.360,0.719221,17.390179,38.705521,3.395195
5423,Zimbabwe,ZWE,2023,16340822,386850.0,5791.192129,2195.225020,62.775,0.768994,15.025473,39.291937,2.926016


In [88]:
for year in [2022, 2023]:
    for gdp_column in GDP_COLUMNS:
        df_full, df_missing = split_by_completeness(df_named, year, gdp_column)
        print(f"{year} - {gdp_column}: {len(df_full)} full countries, {len(df_missing)} missing countries")

2022 - gdp_per_capita_ppp_current_ppp: 180 full countries, 37 missing countries
2022 - gdp_per_capita_usd_current_usd: 181 full countries, 36 missing countries
2023 - gdp_per_capita_ppp_current_ppp: 179 full countries, 38 missing countries
2023 - gdp_per_capita_usd_current_usd: 180 full countries, 37 missing countries


**gdp_per_capita_ppp_current_ppp**

In [89]:
# Year and GDP column studied in the rest of the notebook
year = 2023
gdp_column = 'gdp_per_capita_ppp_current_ppp'

df_full, df_missing = split_by_completeness(df_named, year, gdp_column)
df_missing

,name,ref_area,time_period,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
23,Aruba,ABW,2023,107359,180.000,46574.357420,76.353000,5.002841,NaN,61.835149,NaN
123,Andorra,AND,2023,80856,470.000,71730.668682,84.041000,NaN,8.883022,88.820161,7.699280
223,American Samoa,ASM,2023,47521,200.000,NaN,72.852000,0.002104,4.325937,81.372154,NaN
623,Bermuda,BMU,2023,64698,54.000,114372.049927,82.309000,3.992395,4.634932,100.000000,NaN
873,Channel Islands,CHI,2023,167691,NaN,NaN,81.100000,NaN,NaN,32.755434,NaN
1148,Cuba,CUB,2023,11019931,103800.000,NaN,78.085000,1.817407,9.448016,77.125987,9.383831
1173,Curacao,CUW,2023,154654,444.000,30651.588551,76.800000,NaN,NaN,91.408093,NaN
1198,Cayman Islands,CYM,2023,73038,240.000,88427.786671,80.358000,4.920726,NaN,100.000000,NaN
1473,Eritrea,ERI,2023,3470390,121178.200,NaN,68.624000,0.220206,33.656800,34.696404,3.683768
1648,Faroe Islands,FRO,2023,54398,1370.000,79694.929792,82.952120,0.038604,NaN,39.197527,NaN


In [90]:
# Countries missing one year but not the other (for the selected gdp_column)
missing_2022 = set(split_by_completeness(df_named, 2022, gdp_column)[1]['name'])
missing_2023 = set(split_by_completeness(df_named, 2023, gdp_column)[1]['name'])
print(f"Missing in 2023 but not in 2022: {sorted(missing_2023 - missing_2022)}")
print(f"Missing in 2022 but not in 2023: {sorted(missing_2022 - missing_2023)}")

Missing in 2023 but not in 2022: ['Syrian Arab Republic']
Missing in 2022 but not in 2023: []


Missing countries are : 
- **Group 1:** Small territories or micro-states : Aruba, Bermudes, Guam, Gibraltar, Monaco, Groenland, Hong Kong, Macao,... 
**Loosing them seems to be a wise decision : Many of them aren't souverain states**


- **Group 2:** Countries going through a crisis : Ukraine, Syrie, Yémen, Venezuela, Corée du Nord, Soudan du Sud, Érythrée, Cuba, Cisjordanie et Gaza.
**These are not small countries: ukraine, Yémen, Venezuela or North Korea have millions inhabitants**


A study is needed for the second group

In [91]:
report = missing_population_report(df_named, year, gdp_column)
print(f"Percentage of missing population for {year}: {report['percent_of_world_population'].sum():.2f}%")
report

Percentage of missing population for 2023: 2.60%


,name,population_ps,percent_of_missing_population,percent_of_world_population,land_area_km2,gdp_per_capita_ppp_current_ppp,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
0,"Yemen, Rep.",39390799,18.846349,0.489963,527970.000,NaN,69.295000,0.269743,46.835978,36.570798,9.695592
1,Ukraine,37732836,18.053104,0.469340,579400.000,17665.343750,73.422000,3.225180,14.854702,69.475446,NaN
2,"Venezuela, RB",28300854,13.540415,0.352020,882050.000,NaN,72.514000,3.405915,19.198310,89.249628,3.776114
3,"Korea, Dem. People's Rep.",26418204,12.639670,0.328603,120410.000,NaN,73.642000,2.485362,21.776120,63.538217,NaN
4,Syrian Arab Republic,23594623,11.288740,0.293482,183630.000,NaN,72.120000,1.328993,32.055202,72.167380,2.709988
5,South Sudan,11483374,5.494168,0.142836,631930.000,NaN,57.617000,NaN,23.674692,21.073776,11.610023
6,Cuba,11019931,5.272436,0.137071,103800.000,NaN,78.085000,1.817407,9.448016,77.125987,9.383831
7,"Hong Kong SAR, China",7536100,3.605613,0.093738,1050.000,71512.319355,85.247317,4.527461,NaN,100.000000,NaN
8,Serbia,6623183,3.168832,0.082383,84090.000,29776.895175,76.141463,NaN,19.337551,62.143465,8.031338
9,West Bank and Gaza,5165775,2.471542,0.064255,6025.000,5950.020062,65.170000,NaN,22.394099,86.857687,10.733439


In [92]:
print(f"10 first countries represent {report['percent_of_missing_population'].iloc[:10].sum():.2f}% of the missing population for {year}")

10 first countries represent 94.38% of the missing population for 2023


In [93]:
#Selection of important missing countries with more than 1% of the missing population:

report.iloc[:15]

,name,population_ps,percent_of_missing_population,percent_of_world_population,land_area_km2,gdp_per_capita_ppp_current_ppp,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
0,"Yemen, Rep.",39390799,18.846349,0.489963,527970.0,NaN,69.295000,0.269743,46.835978,36.570798,9.695592
1,Ukraine,37732836,18.053104,0.469340,579400.0,17665.343750,73.422000,3.225180,14.854702,69.475446,NaN
2,"Venezuela, RB",28300854,13.540415,0.352020,882050.0,NaN,72.514000,3.405915,19.198310,89.249628,3.776114
3,"Korea, Dem. People's Rep.",26418204,12.639670,0.328603,120410.0,NaN,73.642000,2.485362,21.776120,63.538217,NaN
4,Syrian Arab Republic,23594623,11.288740,0.293482,183630.0,NaN,72.120000,1.328993,32.055202,72.167380,2.709988
5,South Sudan,11483374,5.494168,0.142836,631930.0,NaN,57.617000,NaN,23.674692,21.073776,11.610023
6,Cuba,11019931,5.272436,0.137071,103800.0,NaN,78.085000,1.817407,9.448016,77.125987,9.383831
7,"Hong Kong SAR, China",7536100,3.605613,0.093738,1050.0,71512.319355,85.247317,4.527461,NaN,100.000000,NaN
8,Serbia,6623183,3.168832,0.082383,84090.0,29776.895175,76.141463,NaN,19.337551,62.143465,8.031338
9,West Bank and Gaza,5165775,2.471542,0.064255,6025.0,5950.020062,65.170000,NaN,22.394099,86.857687,10.733439


**gdp_per_capita_usd_current_usd**

In [98]:
# Year and GDP column studied in the rest of the notebook

year = 2023
gdp_column = 'gdp_per_capita_usd_current_usd'

df_full, df_missing = split_by_completeness(df_named, year, gdp_column)


# Countries missing one year but not the other (for the selected gdp_column)
missing_2022 = set(split_by_completeness(df_named, 2022, gdp_column)[1]['name'])
missing_2023 = set(split_by_completeness(df_named, 2023, gdp_column)[1]['name'])
print(f"Missing in 2023 but not in 2022 for {gdp_column}: {sorted(missing_2023 - missing_2022)}")
print(f"Missing in 2022 but not in 2023 for {gdp_column}: {sorted(missing_2022 - missing_2023)}")


missing_2023_usd = set(split_by_completeness(df_named, 2023, 'gdp_per_capita_usd_current_usd')[1]['name'])
missing_2023_ppp = set(split_by_completeness(df_named, 2023, 'gdp_per_capita_ppp_current_ppp')[1]['name'])
print(f"Missing for ppp but not usd: {sorted(missing_2023_ppp - missing_2023_usd)}")
print(f"Missing in usd but not for ppp: {sorted(missing_2023_usd - missing_2023_ppp)}")



Missing in 2023 but not in 2022 for gdp_per_capita_usd_current_usd: ['Syrian Arab Republic']
Missing in 2022 but not in 2023 for gdp_per_capita_usd_current_usd: []
Missing for ppp but not usd: ['Venezuela, RB']
Missing in usd but not for ppp: []


In [102]:
year = 2023

report_2023_ppp = missing_population_report(df_named, year, 'gdp_per_capita_ppp_current_ppp')
print(f"gdp ppp : Percentage of missing population for {year}: {report_2023_ppp['percent_of_world_population'].sum():.2f}%")
report_2023_ppp[['name', 'gdp_per_capita_ppp_current_ppp']].iloc[:15]


gdp ppp : Percentage of missing population for 2023: 2.60%


,name,gdp_per_capita_ppp_current_ppp
0,"Yemen, Rep.",NaN
1,Ukraine,17665.343750
2,"Venezuela, RB",NaN
3,"Korea, Dem. People's Rep.",NaN
4,Syrian Arab Republic,NaN
5,South Sudan,NaN
6,Cuba,NaN
7,"Hong Kong SAR, China",71512.319355
8,Serbia,29776.895175
9,West Bank and Gaza,5950.020062


In [103]:
year = 2023
report_2023_usd = missing_population_report(df_named, year, 'gdp_per_capita_usd_current_usd')
print(f"gdp usd : Percentage of missing population for {year}: {report_2023_usd['percent_of_world_population'].sum():.2f}%")
report_2023_usd[['name', 'gdp_per_capita_usd_current_usd']].iloc[:15]

gdp usd : Percentage of missing population for 2023: 2.25%


,name,gdp_per_capita_usd_current_usd
0,"Yemen, Rep.",NaN
1,Ukraine,5139.598145
2,"Korea, Dem. People's Rep.",NaN
3,Syrian Arab Republic,NaN
4,South Sudan,NaN
5,Cuba,NaN
6,"Hong Kong SAR, China",50525.111931
7,Serbia,12281.707946
8,West Bank and Gaza,3607.416119
9,Eritrea,NaN


## Decisions — clustering dataset

**Reference year: 2023**
- Most recent year with complete coverage (2024 has structural gaps: no land area, no PM2.5).
- 2022 is closer to the COVID-19 period; 2020–2021 are excluded for that reason.
- Cost: 1 country lost compared with 2022 (*Syrian Arab Republic*).
- Risk: as the latest available year, 2023 values may still be revised by the World Bank.

**GDP per capita: PPP (`gdp_per_capita_ppp_current_ppp`)**
- PPP measures what an income can actually buy locally; GDP in current USD converts at market exchange rates and underestimates living standards in lower-income countries.
- The atlas compares living standards, health and environment, so PPP fits the purpose; GDP in USD would rather measure a country's weight in global markets.
- Cost: 1 country lost compared with USD (*Venezuela, RB*).

**Incomplete countries: complete-case analysis (no imputation)**
- 38 of 217 countries are excluded, leaving 179.
- Neighbor averaging rejected: neighboring countries can differ sharply in political, economic and environmental situation.
- Carrying past values forward rejected: past values are probably missing too (not verified), and they would erase recent shocks such as the war in Ukraine.
- Missingness is not random: excluded countries are mainly countries at war or under sanctions, plus small territories (Hong Kong, Puerto Rico…). Their values are missing because of their situation.
- Limitation: about 2.6% of the world population is excluded, 94% of it in 10 countries, most of them in crisis. The clustering describes the world without these countries.

In [104]:
#Data set final
year = 2023
gdp_column = 'gdp_per_capita_ppp_current_ppp'

df_final, df_missing = split_by_completeness(df_named, year, gdp_column)
print(df_final.shape)
df_final 

(179, 11)


,name,ref_area,time_period,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
48,Afghanistan,AFG,2023,41454761,652230.0,2201.722907,66.035,0.280995,37.948311,25.473053,14.985763
73,Angola,AGO,2023,36749906,1246700.0,9753.600468,64.617,0.743839,19.907569,69.851503,2.546929
98,Albania,ALB,2023,2411658,27400.0,24822.354309,79.602,1.750912,15.514103,58.210609,7.052403
148,United Arab Emirates,ARE,2023,10483751,71020.0,77955.295176,82.909,18.380482,57.186732,85.668070,4.970951
173,Argentina,ARG,2023,45538401,2736690.0,30221.497689,77.395,4.169905,14.346595,92.194513,10.269518
...,...,...,...,...,...,...,...,...,...,...,...
5273,Vanuatu,VUT,2023,320409,12190.0,3567.818820,71.477,0.847979,6.993943,22.265197,4.118668
5298,Samoa,WSM,2023,216663,2780.0,8194.807256,71.698,1.538334,4.579499,17.501280,6.781033
5373,South Africa,ZAF,2023,63212384,1213090.0,15199.732982,66.139,6.759228,27.724711,63.582371,8.909668
5398,Zambia,ZMB,2023,20723965,743390.0,4076.995087,66.349,0.548090,20.243049,45.028585,5.978411


In [105]:
df_countries.head()

,id,iso2Code,name,capitalCity,longitude,latitude,region.id,region.iso2code,region.value,adminregion.id,adminregion.iso2code,adminregion.value,incomeLevel.id,incomeLevel.iso2code,incomeLevel.value,lendingType.id,lendingType.iso2code,lendingType.value
0,ABW,AW,Aruba,Oranjestad,-70.0167,12.51670,LCN,ZJ,Latin America & Caribbean,NaN,NaN,NaN,HIC,XD,High income,LNX,XX,Not classified
2,AFG,AF,Afghanistan,Kabul,69.1761,34.52280,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakis...",LIC,XM,Low income,IDX,XI,IDA
5,AGO,AO,Angola,Luanda,13.2420,-8.81155,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),LMC,XN,Lower middle income,IBD,XF,IBRD
6,ALB,AL,Albania,Tirane,19.8172,41.33170,ECS,Z7,Europe & Central Asia,ECA,7E,Europe & Central Asia (excluding high income),UMC,XT,Upper middle income,IBD,XF,IBRD
7,AND,AD,Andorra,Andorra la Vella,1.5218,42.50750,ECS,Z7,Europe & Central Asia,NaN,NaN,NaN,HIC,XD,High income,LNX,XX,Not classified
